# Treino do classificador de fotos — SP Bem-Estar

Este notebook treina o modelo que o **classifier-worker** usa para conferir se a foto de um relato mostra mesmo o problema informado (buraco, lixo, árvore caída...).

**O que ele faz:**
1. junta fotos de várias fontes (suas fotos no Google Drive e datasets públicos do Roboflow Universe);
2. organiza tudo em uma pasta por categoria e separa treino e validação;
3. treina um **YOLO de classificação** (Ultralytics), partindo de um modelo pré-treinado;
4. mostra a acurácia, a matriz de confusão e as fotos que o modelo errou;
5. exporta para **ONNX**, confere que o resultado em ONNX bate com o do treino usando o mesmo pré-processamento do worker Java, e baixa o arquivo.

**Antes de começar:** no menu, `Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4`. Depois, `Ambiente de execução → Executar tudo`. O treino leva de 20 a 60 minutos, dependendo da quantidade de fotos.

> ⚠️ As Ultralytics são licenciadas sob AGPL-3.0. Para uso comercial, verifique a licença ou troque o modelo base.

## 1. Ambiente

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules
# SMOKE_TEST=1 roda o notebook inteiro em poucos minutos, na CPU, com fotos sintéticas.
# Serve para testar o próprio notebook; no Colab, deixe desligado.
SMOKE_TEST = os.environ.get("SMOKE_TEST") == "1"

if IN_COLAB:
    %pip install -q ultralytics onnx onnxruntime onnxslim roboflow

import torch
import ultralytics

print("Ultralytics", ultralytics.__version__, "| PyTorch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "nenhuma (o treino vai ser lento)")

## 2. Categorias

Os códigos são os mesmos da tabela `category` da API. `OTHER` é a foto que **não** mostra nenhum problema conhecido (rua limpa, selfie, foto de dentro de casa): é ela que permite ao modelo dizer "isso não é um buraco".

A Ultralytics ordena as classes **em ordem alfabética** e grava os nomes dentro do arquivo ONNX. O worker lê esses nomes do próprio modelo, então não é preciso configurar a ordem em lugar nenhum.

Para usar suas fotos, os nomes das pastas podem ser o código (`POTHOLE`) ou um dos apelidos em português (`buraco`).

In [ ]:
CATEGORIES = {
    "POTHOLE":       ("Buraco na via",                            ["buraco", "buracos", "pothole", "potholes"]),
    "TRASH":         ("Descarte irregular de lixo ou entulho",    ["lixo", "entulho", "trash", "garbage"]),
    "TREE":          ("Árvore caída ou com risco de queda",       ["arvore", "árvore", "arvores", "tree", "fallen_tree"]),
    "FLOODING":      ("Alagamento",                               ["alagamento", "enchente", "flood", "flooding"]),
    "SEWAGE":        ("Esgoto a céu aberto",                      ["esgoto", "sewage"]),
    "CLOGGED_DRAIN": ("Bueiro ou boca de lobo entupida",          ["bueiro", "boca_de_lobo", "drain"]),
    "SIDEWALK":      ("Calçada danificada ou sem acessibilidade", ["calcada", "calçada", "sidewalk"]),
    "STREETLIGHT":   ("Poste ou iluminação pública com defeito",  ["poste", "iluminacao", "streetlight"]),
    "BUS_STOP":      ("Ponto de ônibus danificado",               ["ponto_de_onibus", "ponto", "bus_stop"]),
    "OVERGROWN":     ("Mato alto ou terreno sem capinação",       ["mato", "capinacao", "overgrown"]),
    "OTHER":         ("Nenhum problema conhecido",                ["outro", "outros", "nenhum", "other", "normal"]),
}


def normalize(name):
    import unicodedata
    name = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode().lower()
    return name.strip().replace(" ", "_").replace("-", "_")


ALIASES = {}
for code_, (_, aliases) in CATEGORIES.items():
    for alias in [code_] + aliases:
        ALIASES[normalize(alias)] = code_


def category_of(folder_name):
    return ALIASES.get(normalize(folder_name))


print(f"{len(CATEGORIES)} categorias:", ", ".join(CATEGORIES))

## 3. Fontes das fotos

Use quantas fontes quiser; as fotos são juntadas e as repetidas, descartadas.

**A) Suas fotos no Google Drive (recomendado).** Monte um `.zip` com uma pasta por categoria e suba no Drive:
```
fotos.zip
├── buraco/      (ou POTHOLE/)
├── alagamento/
├── outro/       ← fotos de rua sem problema nenhum
└── ...
```
Fotos do próprio cenário (ruas de São Paulo, tiradas com celular) ensinam mais do que qualquer dataset estrangeiro. Para categorias sem dataset público (bueiro, esgoto, ponto de ônibus), essa é a principal fonte.

**B) Roboflow Universe.** Datasets públicos em [universe.roboflow.com](https://universe.roboflow.com). Precisa de uma chave de API gratuita: crie em `app.roboflow.com → Settings → API Keys` e guarde nos **Segredos** do Colab (ícone de chave à esquerda) com o nome `ROBOFLOW_API_KEY`.
- Projetos de **detecção**: toda imagem com pelo menos uma marcação vira exemplo da categoria; imagens sem marcação podem virar `OTHER`.
- Projetos de **classificação**: informe como cada classe do projeto vira uma categoria (`mapping`).

**C) Outros datasets** (Kaggle, Mendeley, artigos): baixe, separe nas pastas e inclua no zip da opção A. Exemplo: o [Flood10k](https://data.mendeley.com/datasets/4sgcd7zdsp/1) tem milhares de fotos de alagamento e de ruas normais.

> Confira a licença de cada dataset antes de usar o modelo fora de estudo.

In [ ]:
# A) .zip no Google Drive. None para não usar.
DRIVE_ZIP = "/content/drive/MyDrive/sp-bem-estar/fotos.zip"

# B) Projetos do Roboflow Universe: "workspace/projeto" está na URL do projeto.
#    target: categoria das imagens com marcação (projetos de detecção)
#    mapping: classe do projeto -> categoria (projetos de classificação)
#    negatives_to_other: imagens sem marcação viram OTHER
ROBOFLOW_SOURCES = [
    {"project": "fallen-tree-on-roads/fallen-trees-on-road", "target": "TREE"},
    {"project": "object-qf9bc/pothole-1nmg1", "target": "POTHOLE", "negatives_to_other": True},
]

MAX_PER_SOURCE = 1500   # limita fontes muito grandes para não desequilibrar as categorias
MIN_PER_CLASS = 100     # abaixo disso a categoria é avisada como fraca
MIN_TO_TRAIN = 20       # abaixo disso a categoria fica fora do treino

WORK_DIR = "/content/sp-bem-estar" if IN_COLAB else os.environ.get("WORK_DIR", "work")

In [ ]:
import hashlib
import random
import shutil
import zipfile
from collections import Counter
from pathlib import Path

from PIL import Image, ImageOps

WORK = Path(WORK_DIR)
RAW = WORK / "raw"            # fotos já validadas, uma pasta por categoria
DATASET = WORK / "dataset"    # train/ e val/ no formato da Ultralytics
DOWNLOADS = WORK / "downloads"
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

_seen_hashes = set()


def add_image(src, category, source):
    # Valida, corrige a rotação do celular (EXIF), converte para RGB JPEG e descarta repetidas
    try:
        with Image.open(src) as img:
            img = ImageOps.exif_transpose(img).convert("RGB")
    except Exception:
        return False
    if min(img.size) < 64:
        return False
    img.thumbnail((1024, 1024))  # o modelo usa 224 px; isso só economiza disco
    target = RAW / category
    target.mkdir(parents=True, exist_ok=True)
    tmp = target / f".tmp-{source}.jpg"
    img.save(tmp, "JPEG", quality=92)
    digest = hashlib.sha1(tmp.read_bytes()).hexdigest()
    if digest in _seen_hashes:
        tmp.unlink()
        return False
    _seen_hashes.add(digest)
    tmp.rename(target / f"{source}-{digest[:16]}.jpg")
    return True


def images_in(folder):
    return sorted(p for p in Path(folder).rglob("*") if p.suffix.lower() in IMAGE_EXTENSIONS)


def import_class_folders(root, source, mapping=None):
    # Pastas com nome de categoria (ou apelido). mapping sobrepõe o nome da pasta.
    counts = Counter()
    for folder in sorted(p for p in Path(root).rglob("*") if p.is_dir()):
        category = (mapping or {}).get(folder.name) or category_of(folder.name)
        if category is None:
            continue
        files = [f for f in folder.iterdir() if f.suffix.lower() in IMAGE_EXTENSIONS]
        random.Random(0).shuffle(files)
        for f in files[:MAX_PER_SOURCE]:
            counts[category] += add_image(f, category, source)
    return counts


def import_yolo_detection(root, source, target, negatives_to_other=False):
    # Export "yolov8" do Roboflow: */images/x.jpg + */labels/x.txt (vazio = sem marcação)
    counts = Counter()
    images = [p for p in images_in(root) if p.parent.name == "images"]
    random.Random(0).shuffle(images)
    for img in images:
        label = img.parent.parent / "labels" / (img.stem + ".txt")
        has_objects = label.exists() and label.read_text().strip() != ""
        category = target if has_objects else ("OTHER" if negatives_to_other else None)
        if category and counts[category] < MAX_PER_SOURCE:
            counts[category] += add_image(img, category, source)
    return counts

In [ ]:
def from_drive_zip(path):
    if IN_COLAB and path.startswith("/content/drive"):
        from google.colab import drive
        drive.mount("/content/drive")
    if not Path(path).exists():
        print(f"Zip não encontrado em {path}; pulando.")
        return Counter()
    target = DOWNLOADS / "drive"
    with zipfile.ZipFile(path) as z:
        z.extractall(target)
    return import_class_folders(target, "drive")


def roboflow_api_key():
    if IN_COLAB:
        from google.colab import userdata
        try:
            return userdata.get("ROBOFLOW_API_KEY")
        except Exception:
            return None
    return os.environ.get("ROBOFLOW_API_KEY")


def from_roboflow(spec, api_key):
    from roboflow import Roboflow

    workspace, name = spec["project"].split("/")
    project = Roboflow(api_key=api_key).workspace(workspace).project(name)
    version = max(project.versions(), key=lambda v: int(str(v.version).split("/")[-1]))
    source = f"rf-{name}"
    if project.type == "classification":
        dataset = version.download("folder", location=str(DOWNLOADS / source), overwrite=True)
        return import_class_folders(dataset.location, source, spec.get("mapping"))
    dataset = version.download("yolov8", location=str(DOWNLOADS / source), overwrite=True)
    return import_yolo_detection(dataset.location, source, spec["target"], spec.get("negatives_to_other", False))


def synthetic_photos(per_class=40):
    # Só para SMOKE_TEST: cada categoria tem uma cor dominante, com ruído
    import numpy as np

    rng = np.random.default_rng(0)
    counts = Counter()
    colors = rng.integers(0, 256, size=(len(CATEGORIES), 3))
    for (category, color) in zip(CATEGORIES, colors):
        for i in range(per_class):
            w, h = rng.integers(120, 400, size=2)
            pixels = np.clip(color + rng.normal(0, 25, size=(h, w, 3)), 0, 255).astype("uint8")
            path = DOWNLOADS / "synthetic" / f"{category}-{i}.png"
            path.parent.mkdir(parents=True, exist_ok=True)
            Image.fromarray(pixels).save(path)
            counts[category] += add_image(path, category, "synthetic")
    return counts

## 4. Juntar as fotos

In [ ]:
shutil.rmtree(WORK, ignore_errors=True)
totals = Counter()

if SMOKE_TEST:
    totals += synthetic_photos()
else:
    if DRIVE_ZIP:
        found = from_drive_zip(DRIVE_ZIP)
        print("Drive:", dict(found))
        totals += found
    api_key = roboflow_api_key() if ROBOFLOW_SOURCES else None
    if ROBOFLOW_SOURCES and not api_key:
        print("Sem ROBOFLOW_API_KEY nos Segredos do Colab; pulando os datasets do Roboflow.")
    for spec in ROBOFLOW_SOURCES if api_key else []:
        try:
            found = from_roboflow(spec, api_key)
            print(f"Roboflow {spec['project']}:", dict(found))
            totals += found
        except Exception as e:
            print(f"Roboflow {spec['project']}: falhou ({e}); pulando.")

print()
print(f"{'categoria':<15}{'fotos':>7}")
for category in CATEGORIES:
    n = totals[category]
    note = "  ← fora do treino" if n < MIN_TO_TRAIN else ("  ← poucas fotos" if n < MIN_PER_CLASS else "")
    print(f"{category:<15}{n:>7}{note}")

TRAIN_CLASSES = [c for c in CATEGORIES if totals[c] >= MIN_TO_TRAIN]
assert "OTHER" in TRAIN_CLASSES, "Sem fotos de OTHER o modelo não aprende a dizer 'isso não é um problema'."
assert len(TRAIN_CLASSES) >= 2, "São precisas ao menos duas categorias com fotos."
print("\nCategorias no treino:", ", ".join(TRAIN_CLASSES))

Confira algumas fotos de cada categoria. Uma foto na pasta errada ensina o modelo errado — vale a pena olhar.

In [ ]:
import matplotlib.pyplot as plt

per_row = 6
fig, axes = plt.subplots(len(TRAIN_CLASSES), per_row, figsize=(per_row * 2, len(TRAIN_CLASSES) * 2))
for row, category in zip(axes if len(TRAIN_CLASSES) > 1 else [axes], TRAIN_CLASSES):
    samples = sorted((RAW / category).glob("*.jpg"))
    random.Random(1).shuffle(samples)
    for ax, path in zip(row, samples[:per_row] + [None] * per_row):
        ax.axis("off")
        if path:
            ax.imshow(Image.open(path))
    row[0].set_title(category, loc="left", fontsize=10)
plt.tight_layout()
plt.show()

## 5. Separar treino e validação

80% das fotos de cada categoria vão para o treino e 20% para a validação, que o modelo nunca vê durante o treino e serve para medir a acurácia de verdade.

In [ ]:
VAL_FRACTION = 0.2

for split in ("train", "val"):
    shutil.rmtree(DATASET / split, ignore_errors=True)

for category in TRAIN_CLASSES:
    files = sorted((RAW / category).glob("*.jpg"))
    random.Random(42).shuffle(files)
    n_val = max(1, round(len(files) * VAL_FRACTION))
    for split, chunk in (("val", files[:n_val]), ("train", files[n_val:])):
        target = DATASET / split / category
        target.mkdir(parents=True, exist_ok=True)
        for f in chunk:
            shutil.copy(f, target / f.name)

for split in ("train", "val"):
    counts = {c: len(list((DATASET / split / c).glob("*.jpg"))) for c in TRAIN_CLASSES}
    print(f"{split:<6}", counts)

## 6. Treinar

`yolo11n-cls` é o menor e mais rápido; `yolo11s-cls` costuma acertar um pouco mais e leva umas 3 vezes mais tempo. Os dois rodam bem na CPU do worker.

`patience` interrompe o treino quando a validação para de melhorar.

In [ ]:
from ultralytics import YOLO

BASE_MODEL = "yolo11n-cls.pt"
EPOCHS = 40
IMAGE_SIZE = 224   # o mesmo tamanho que o worker usa

if SMOKE_TEST:
    model = YOLO("yolo11n-cls.yaml")   # sem pesos pré-treinados: dispensa download
    train_args = dict(epochs=2, batch=16, workers=0, device="cpu", plots=False)
else:
    model = YOLO(BASE_MODEL)
    train_args = dict(epochs=EPOCHS, batch=64, patience=10)

results = model.train(data=str(DATASET), imgsz=IMAGE_SIZE, seed=42, deterministic=True,
                      project=str(WORK / "runs"), name="sp", exist_ok=True, **train_args)
BEST = Path(results.save_dir) / "weights" / "best.pt"
print("Melhor modelo:", BEST)

## 7. Avaliar

- **top-1**: em quantas fotos de validação a categoria mais provável é a certa.
- **acerto por categoria**: mostra onde o modelo ainda é fraco — normalmente as categorias com menos fotos.
- **confusões**: pares de categorias que o modelo mistura (alagamento × esgoto, por exemplo).

In [ ]:
import numpy as np

best = YOLO(BEST)
metrics = best.val(data=str(DATASET), imgsz=IMAGE_SIZE, split="val", plots=not SMOKE_TEST,
                   workers=0 if SMOKE_TEST else 8, verbose=False,
                   project=str(WORK / "runs"), name="val", exist_ok=True)
names = best.names   # {índice: nome}, em ordem alfabética
print(f"top-1: {metrics.top1:.1%}   top-5: {metrics.top5:.1%}")

# matrix[previsto][real]
matrix = metrics.confusion_matrix.matrix[:len(names), :len(names)]
per_class = {}
print(f"\n{'categoria':<15}{'acerto':>8}{'fotos':>7}")
for i, name in names.items():
    total = matrix[:, i].sum()
    per_class[name] = float(matrix[i, i] / total) if total else 0.0
    print(f"{name:<15}{per_class[name]:>8.0%}{int(total):>7}")

confusions = [(matrix[p, t], names[t], names[p]) for t in names for p in names if p != t and matrix[p, t] > 0]
print("\nConfusões mais comuns (real → previsto):")
for n, real, predicted in sorted(confusions, reverse=True)[:8]:
    print(f"  {real} → {predicted}: {int(n)}")

In [ ]:
# Fotos de validação que o modelo errou
val_images = sorted((DATASET / "val").rglob("*.jpg"))
wrong = []
for batch_start in range(0, len(val_images), 64):
    chunk = val_images[batch_start:batch_start + 64]
    for path, result in zip(chunk, best.predict([str(p) for p in chunk], imgsz=IMAGE_SIZE, verbose=False)):
        predicted = names[int(result.probs.top1)]
        if predicted != path.parent.name:
            wrong.append((path, path.parent.name, predicted, float(result.probs.top1conf)))

print(f"{len(wrong)} de {len(val_images)} fotos de validação erradas")
if wrong:
    shown = wrong[:12]
    fig, axes = plt.subplots(1, len(shown), figsize=(len(shown) * 2.2, 2.6))
    for ax, (path, real, predicted, conf) in zip(np.atleast_1d(axes), shown):
        ax.imshow(Image.open(path))
        ax.set_title(f"{real}\n→ {predicted} ({conf:.0%})", fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## 8. Exportar para ONNX e conferir

A célula abaixo exporta o modelo e roda o arquivo ONNX com o **mesmo pré-processamento do worker Java** (menor lado em 224, recorte central, RGB em 0..1). Se o ONNX e o modelo do treino discordarem em muitas fotos, alguma coisa no pré-processamento divergiu e o worker não teria a acurácia medida acima.

In [ ]:
import ast
import onnxruntime as ort

onnx_path = Path(best.export(format="onnx", imgsz=IMAGE_SIZE, simplify=True, dynamic=False, opset=17))
session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])

# Os rótulos ficam no metadado "names"; é dele que o worker lê
labels_in_model = ast.literal_eval(session.get_modelmeta().custom_metadata_map["names"])
LABELS = [labels_in_model[i] for i in range(len(labels_in_model))]
assert LABELS == [names[i] for i in range(len(names))]
print("Rótulos no modelo:", LABELS)


def java_preprocess(path, size=IMAGE_SIZE):
    # Mesmo cálculo de ImagePreprocessor.java: redimensiona o menor lado e recorta o centro
    img = Image.open(path).convert("RGB")
    scale = size / min(img.size)
    w, h = max(size, round(img.width * scale)), max(size, round(img.height * scale))
    img = img.resize((w, h), Image.BILINEAR)
    left, top = (w - size) // 2, (h - size) // 2
    img = img.crop((left, top, left + size, top + size))
    return (np.asarray(img, dtype=np.float32) / 255.0).transpose(2, 0, 1)[None]


input_name = session.get_inputs()[0].name
agree = 0
for path in val_images:
    onnx_top1 = int(np.argmax(session.run(None, {input_name: java_preprocess(path)})[0][0]))
    torch_top1 = int(best.predict(str(path), imgsz=IMAGE_SIZE, verbose=False)[0].probs.top1)
    agree += onnx_top1 == torch_top1
agreement = agree / len(val_images)
print(f"ONNX (pré-processamento do worker) concorda com o modelo do treino em {agreement:.1%} das fotos")
if agreement < 0.95:
    print("⚠️ Concordância baixa: o worker vai acertar menos do que a validação indica.")

## 9. Salvar e baixar

In [ ]:
import datetime
import json

VERSION = datetime.date.today().strftime("%Y-%m-%d") + "-" + Path(BASE_MODEL).stem
OUTPUT = WORK / "output"
OUTPUT.mkdir(parents=True, exist_ok=True)
shutil.copy(onnx_path, OUTPUT / "classifier.onnx")
(OUTPUT / "labels.txt").write_text("\n".join(LABELS) + "\n")
(OUTPUT / "metrics.json").write_text(json.dumps({
    "version": VERSION,
    "base_model": BASE_MODEL,
    "ultralytics": ultralytics.__version__,
    "top1": float(metrics.top1),
    "top5": float(metrics.top5),
    "per_class_accuracy": per_class,
    "onnx_agreement": agreement,
    "images": {c: totals[c] for c in TRAIN_CLASSES},
}, indent=2, ensure_ascii=False))
confusion_png = Path(metrics.save_dir) / "confusion_matrix_normalized.png"
if confusion_png.exists():
    shutil.copy(confusion_png, OUTPUT / "confusion_matrix.png")

print("Arquivos em", OUTPUT, ":", sorted(p.name for p in OUTPUT.iterdir()))
print("\nVersão do modelo:", VERSION)

if IN_COLAB:
    from google.colab import files
    if Path("/content/drive/MyDrive").exists():
        drive_out = Path("/content/drive/MyDrive/sp-bem-estar/modelos") / VERSION
        shutil.copytree(OUTPUT, drive_out, dirs_exist_ok=True)
        print("Cópia no Drive:", drive_out)
    files.download(str(OUTPUT / "classifier.onnx"))
    files.download(str(OUTPUT / "metrics.json"))

## 10. Usar no worker

1. Coloque o arquivo baixado em `classifier-worker/models/classifier.onnx` (a pasta fica fora do git).
2. Rode o worker informando a versão impressa acima, que vai junto com cada resultado:
   ```bash
   CLASSIFIER_MODEL_VERSION=<versão> java -jar classifier-worker/target/classifier-worker-0.0.1-SNAPSHOT.jar
   ```
   Os rótulos vêm de dentro do modelo; não é preciso configurar `classifier.labels`.
3. Guarde o `metrics.json` junto do modelo: ele registra com que fotos e com que acurácia aquela versão foi treinada.

Para melhorar o modelo, junte mais fotos das categorias com menor acerto e rode o notebook de novo.